
# Stress Model V3 — Google Drive Dataset + Physiology-Preserving EEG/GSR Fusion

**Goal:** improve unseen-subject stress classification while keeping the deployment input as exactly:

- **1 EEG image**
- **1 GSR image**

This notebook changes the representation rather than only changing activation functions:

- 30-second windows instead of 5-second windows
- EEG image preserves **all 32 channels**, time evolution, and theta/alpha/beta band power
- GSR image preserves **tonic level, phasic response, and derivative**
- strict subject-wise Train / Validation / Test split is preserved
- dual compact CNN + squeeze-excitation + gated feature fusion
- optional 3-seed ensemble if one model is still below target
- Keras + TFLite export

> **Important:** 80%+ unseen-subject accuracy is a target, not a guaranteed result. The notebook keeps the evaluation scientifically valid instead of introducing subject leakage just to inflate accuracy.


**Dataset source:** `stress_dataset_colab.zip` is loaded directly from Google Drive; no browser upload is required.

In [ ]:

# ============================================================
# 1. COLAB SETUP — LOAD DATASET ZIP FROM GOOGLE DRIVE
# ============================================================

import zipfile
import shutil
from pathlib import Path
from google.colab import drive

print("=" * 72)
print("MOUNTING GOOGLE DRIVE")
print("=" * 72)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
ZIP_NAME = "stress_dataset_colab.zip"

print("\nSearching Google Drive for:")
print(ZIP_NAME)

matches = list(DRIVE_ROOT.rglob(ZIP_NAME))

if len(matches) == 0:
    raise FileNotFoundError(
        f"{ZIP_NAME} was not found anywhere inside MyDrive.\n"
        "Place the ZIP anywhere in MyDrive and rerun this cell."
    )

if len(matches) > 1:
    print("\nMultiple matching ZIP files found:")
    for i, p in enumerate(matches, start=1):
        print(f"{i}. {p}")
    print("\nUsing the first match.")

DRIVE_ZIP_PATH = matches[0]

print("\nDataset ZIP found:")
print(DRIVE_ZIP_PATH)

size_mb = DRIVE_ZIP_PATH.stat().st_size / (1024 ** 2)
print(f"ZIP size: {size_mb:.2f} MB")

if not zipfile.is_zipfile(DRIVE_ZIP_PATH):
    raise RuntimeError(
        "The Google Drive file is not a valid ZIP or is incomplete."
    )

print("ZIP format: VALID ✅")

DATASET_DIR = Path("/content/stress_dataset")
MANIFEST_PATH = DATASET_DIR / "window_manifest.csv"
WINDOW_DIR = DATASET_DIR / "windows"

dataset_ready = (
    MANIFEST_PATH.exists()
    and WINDOW_DIR.exists()
    and len(list(WINDOW_DIR.glob("*.npz"))) == 35
)

if dataset_ready:
    print("\nExisting extracted dataset detected.")
    print("Skipping extraction.")
else:
    if DATASET_DIR.exists():
        print("\nRemoving incomplete/old extraction...")
        shutil.rmtree(DATASET_DIR)

    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    print("\nExtracting dataset from Google Drive...")

    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
        bad_file = z.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"Corrupted ZIP entry detected: {bad_file}"
            )

        z.extractall(DATASET_DIR)

    print("Extraction complete ✅")

npz_files = sorted(WINDOW_DIR.glob("*.npz"))

print("\n" + "=" * 72)
print("DATASET VERIFICATION")
print("=" * 72)

print("Manifest exists:", MANIFEST_PATH.exists())
print("Windows folder :", WINDOW_DIR.exists())
print("NPZ files      :", len(npz_files))

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing manifest: {MANIFEST_PATH}"
    )

if len(npz_files) != 35:
    raise RuntimeError(
        f"Expected 35 NPZ subject files, found {len(npz_files)}."
    )

print("\nORIGINAL DATASET READY ✅")
print("You can now run the V3 preprocessing cell.")


In [ ]:

# ============================================================
# 2. V3 PREPROCESSING
#    30-SECOND EEG BAND-POWER IMAGE + GSR PHYSIOLOGY IMAGE
# ============================================================

import gc
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import spectrogram, savgol_filter
from PIL import Image, ImageDraw

DATASET_ROOT = Path("/content/stress_dataset")
WINDOW_DIR = DATASET_ROOT / "windows"
MANIFEST_PATH = DATASET_ROOT / "window_manifest.csv"

OUT_ROOT = Path("/content/stress_image_dataset_v3")
EEG_DIR = OUT_ROOT / "eeg"
GSR_DIR = OUT_ROOT / "gsr"
V3_MANIFEST = OUT_ROOT / "image_manifest_v3.csv"

EEG_DIR.mkdir(parents=True, exist_ok=True)
GSR_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
EEG_FS = 128
GSR_FS = 4

# Existing data = 5-second windows.
# 6 windows = 30 seconds.
SEGMENT_WINDOWS = 6

# stride=2 => 10-second step between 30-second segments.
SEGMENT_STRIDE = 2

LABEL_NAMES = ["low", "moderate", "high"]

base_manifest = pd.read_csv(MANIFEST_PATH)

print("=" * 76)
print("V3 PHYSIOLOGY-PRESERVING IMAGE GENERATION")
print("=" * 76)
print("Subjects:", base_manifest["subject"].nunique())
print("Original 5-sec samples:", len(base_manifest))

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def decode_phase(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)

def robust_uint8(x, lo=None, hi=None):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if lo is None:
        lo = np.percentile(x, 2)
    if hi is None:
        hi = np.percentile(x, 98)

    if hi <= lo:
        hi = lo + 1e-6

    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (x * 255).astype(np.uint8)

def eeg_band_image(eeg_30s):
    """
    eeg_30s shape: (32, 3840)
    Output: 32 x T x 3 where RGB = theta / alpha / beta log-power.
    This preserves every EEG channel instead of averaging 32 channels.
    """
    bands = [(4, 8), (8, 13), (13, 30)]
    channel_features = []

    for ch in range(eeg_30s.shape[0]):
        f, t, sxx = spectrogram(
            eeg_30s[ch],
            fs=EEG_FS,
            nperseg=256,
            noverlap=192,
            nfft=256,
            scaling="density",
            mode="psd"
        )

        band_series = []
        for low, high in bands:
            mask = (f >= low) & (f < high)
            p = np.mean(sxx[mask], axis=0)
            p = 10.0 * np.log10(p + 1e-12)
            band_series.append(p.astype(np.float32))

        channel_features.append(np.stack(band_series, axis=-1))

    # (32, time, 3)
    feat = np.stack(channel_features, axis=0)

    # Robustly normalize each physiological band independently.
    rgb = np.zeros_like(feat, dtype=np.uint8)
    for b in range(3):
        rgb[..., b] = robust_uint8(feat[..., b])

    img = Image.fromarray(rgb)
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC)
    return img

def smooth_gsr(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) >= 11:
        return savgol_filter(x, 11, 2, mode="interp").astype(np.float32)
    return x.copy()

def draw_signal_channel(signal, global_lo, global_hi):
    """
    Fixed-scale raster. Unlike per-window z-normalization,
    absolute tonic/amplitude information is retained.
    """
    signal = np.asarray(signal, dtype=np.float32)
    signal = np.nan_to_num(signal, nan=0.0, posinf=global_hi, neginf=global_lo)

    if global_hi <= global_lo:
        global_hi = global_lo + 1e-6

    y = np.clip((signal - global_lo) / (global_hi - global_lo), 0, 1)
    xs = np.linspace(0, IMAGE_SIZE - 1, len(y))
    ys = (1.0 - y) * (IMAGE_SIZE - 1)

    canvas = Image.new("L", (IMAGE_SIZE, IMAGE_SIZE), 0)
    draw = ImageDraw.Draw(canvas)

    pts = [(float(xs[i]), float(ys[i])) for i in range(len(y))]
    if len(pts) >= 2:
        draw.line(pts, fill=255, width=3)

    return np.asarray(canvas, dtype=np.uint8)

def gsr_three_channel_image(gsr_30s, scale):
    """
    RGB channels:
      R = raw GSR (tonic + phasic)
      G = phasic component
      B = first derivative
    """
    raw = np.asarray(gsr_30s, dtype=np.float32)
    tonic = smooth_gsr(raw)
    phasic = raw - tonic
    deriv = np.gradient(raw).astype(np.float32)

    r = draw_signal_channel(raw, scale["raw_lo"], scale["raw_hi"])
    g = draw_signal_channel(phasic, scale["phasic_lo"], scale["phasic_hi"])
    b = draw_signal_channel(deriv, scale["deriv_lo"], scale["deriv_hi"])

    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray(rgb)

def build_subject_segments(npz_file):
    """
    Returns metadata needed to reconstruct 30-second segments.
    Segments never cross phase/label boundaries.
    """
    data = np.load(npz_file, mmap_mode="r")
    labels = np.asarray(data["labels"]).astype(np.int32)
    phases = np.asarray([decode_phase(x) for x in data["phases"]])
    window_indices = np.asarray(data["window_indices"]).astype(np.int32)

    segments = []

    for phase in np.unique(phases):
        idx = np.where(phases == phase)[0]
        idx = idx[np.argsort(window_indices[idx])]

        # Require same label within the phase
        for start in range(0, len(idx) - SEGMENT_WINDOWS + 1, SEGMENT_STRIDE):
            sel = idx[start:start + SEGMENT_WINDOWS]

            # Ensure truly consecutive original windows
            wis = window_indices[sel]
            if not np.all(np.diff(wis) == 1):
                continue

            labs = labels[sel]
            if not np.all(labs == labs[0]):
                continue

            segments.append({
                "phase": phase,
                "start_window": int(wis[0]),
                "indices": sel,
                "label_id": int(labs[0])
            })

    data.close()
    return segments

# ------------------------------------------------------------
# Compute TRAIN-ONLY GSR scaling
# ------------------------------------------------------------

print("\nComputing train-only GSR scaling...")

train_subjects = set(
    base_manifest.loc[base_manifest["split"] == "train", "subject"].astype(str).unique()
)

raw_vals = []
phasic_vals = []
deriv_vals = []

for npz_file in sorted(WINDOW_DIR.glob("*.npz")):
    subject = npz_file.stem
    if subject not in train_subjects:
        continue

    data = np.load(npz_file, mmap_mode="r")
    eda = data["eda"]

    # sample every few windows to keep memory tiny
    for i in range(0, len(eda), 10):
        x = np.asarray(eda[i], dtype=np.float32)
        tonic = smooth_gsr(x)
        raw_vals.append(x)
        phasic_vals.append(x - tonic)
        deriv_vals.append(np.gradient(x).astype(np.float32))

    data.close()

raw_vals = np.concatenate(raw_vals)
phasic_vals = np.concatenate(phasic_vals)
deriv_vals = np.concatenate(deriv_vals)

GSR_SCALE = {
    "raw_lo": float(np.percentile(raw_vals, 1)),
    "raw_hi": float(np.percentile(raw_vals, 99)),
    "phasic_lo": float(np.percentile(phasic_vals, 1)),
    "phasic_hi": float(np.percentile(phasic_vals, 99)),
    "deriv_lo": float(np.percentile(deriv_vals, 1)),
    "deriv_hi": float(np.percentile(deriv_vals, 99)),
}

print("GSR scale:", GSR_SCALE)

del raw_vals, phasic_vals, deriv_vals
gc.collect()

# ------------------------------------------------------------
# Generate paired 30-sec images
# ------------------------------------------------------------

records = []
npz_files = sorted(WINDOW_DIR.glob("*.npz"))

for sidx, npz_file in enumerate(npz_files, start=1):
    subject = npz_file.stem
    subject_rows = base_manifest[base_manifest["subject"].astype(str) == subject]

    if len(subject_rows) == 0:
        continue

    split = str(subject_rows["split"].iloc[0])

    eeg_subdir = EEG_DIR / subject
    gsr_subdir = GSR_DIR / subject
    eeg_subdir.mkdir(parents=True, exist_ok=True)
    gsr_subdir.mkdir(parents=True, exist_ok=True)

    data = np.load(npz_file, mmap_mode="r")
    eeg = data["eeg"]
    eda = data["eda"]

    segments = build_subject_segments(npz_file)

    new_count = 0
    for seg in segments:
        phase = seg["phase"]
        start_w = seg["start_window"]
        label_id = seg["label_id"]
        sel = seg["indices"]

        stem = f"{subject}_{phase}_s{start_w:03d}"
        eeg_path = eeg_subdir / f"{stem}.png"
        gsr_path = gsr_subdir / f"{stem}.png"

        if not (eeg_path.exists() and gsr_path.exists()):
            # (6,32,640) -> (32,3840)
            eeg_30s = np.concatenate(
                [np.asarray(eeg[i], dtype=np.float32) for i in sel],
                axis=1
            )

            # (6,20) -> (120,)
            gsr_30s = np.concatenate(
                [np.asarray(eda[i], dtype=np.float32) for i in sel],
                axis=0
            )

            eeg_img = eeg_band_image(eeg_30s)
            gsr_img = gsr_three_channel_image(gsr_30s, GSR_SCALE)

            eeg_img.save(eeg_path, format="PNG", optimize=True)
            gsr_img.save(gsr_path, format="PNG", optimize=True)
            new_count += 1

        records.append({
            "subject": subject,
            "phase": phase,
            "start_window": start_w,
            "label_id": label_id,
            "label": LABEL_NAMES[label_id],
            "split": split,
            "eeg_image": str(eeg_path),
            "gsr_image": str(gsr_path),
        })

    data.close()
    gc.collect()

    print(f"[{sidx:02d}/{len(npz_files)}] {subject}: segments={len(segments)}, new={new_count}")

v3 = pd.DataFrame(records)
v3.to_csv(V3_MANIFEST, index=False)

print("\n" + "=" * 76)
print("V3 IMAGE DATASET COMPLETE")
print("=" * 76)
print("Segments:", len(v3))
print("Subjects:", v3["subject"].nunique())
print("\nClasses:")
print(v3["label"].value_counts())
print("\nSplits:")
print(v3["split"].value_counts())
print("\nManifest:", V3_MANIFEST)

# Strict leakage check
assert v3.groupby("subject")["split"].nunique().max() == 1
assert v3["eeg_image"].map(os.path.exists).all()
assert v3["gsr_image"].map(os.path.exists).all()

print("\nNO SUBJECT LEAKAGE ✅")
print("PAIRED 30-SECOND IMAGE DATASET READY ✅")


In [ ]:

# ============================================================
# 3. VISUAL SANITY CHECK
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

v3 = pd.read_csv("/content/stress_image_dataset_v3/image_manifest_v3.csv")

sample = v3.sample(3, random_state=42).reset_index(drop=True)

for i, row in sample.iterrows():
    eeg_img = Image.open(row["eeg_image"])
    gsr_img = Image.open(row["gsr_image"])

    plt.figure(figsize=(5, 4))
    plt.imshow(eeg_img)
    plt.title(f"EEG | {row['subject']} | {row['label']} | {row['phase']}")
    plt.axis("off")
    plt.show()

    plt.figure(figsize=(5, 4))
    plt.imshow(gsr_img)
    plt.title(f"GSR | {row['subject']} | {row['label']} | {row['phase']}")
    plt.axis("off")
    plt.show()

print("Visual check complete.")


In [ ]:

# ============================================================
# 4. TRAIN DUAL CNN + SQUEEZE-EXCITATION + GATED FUSION
# ============================================================

import os, json, random, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
BATCH = 32
EPOCHS = 40
LABELS = ["low", "moderate", "high"]

MANIFEST = Path("/content/stress_image_dataset_v3/image_manifest_v3.csv")
MODEL_DIR = Path("/content/stress_model_v3")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("V3 DUAL PHYSIOLOGY IMAGE MODEL")
print("=" * 76)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Enable T4 GPU before training.")

df = pd.read_csv(MANIFEST)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

assert df.groupby("subject")["split"].nunique().max() == 1

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

def read_image(path):
    x = tf.io.read_file(path)
    x = tf.io.decode_png(x, channels=3)
    x = tf.image.resize(x, [IMG_SIZE, IMG_SIZE])
    x = tf.cast(x, tf.float32) / 255.0
    return x

def load_pair(eeg_path, gsr_path, label):
    return {
        "eeg_image": read_image(eeg_path),
        "gsr_image": read_image(gsr_path),
    }, label

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame["eeg_image"].astype(str).values,
        frame["gsr_image"].astype(str).values,
        frame["label_id"].astype(np.int32).values
    ))

    if training:
        ds = ds.shuffle(min(len(frame), 5000), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)
test_ds = make_ds(test_df, False)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label_id"].values
)
class_weights = {i: float(weights[i]) for i in range(3)}
print("Class weights:", class_weights)

# ------------------------------------------------------------
# Model blocks
# ------------------------------------------------------------

def se_block(x, ratio=8, name=None):
    channels = int(x.shape[-1])
    s = tf.keras.layers.GlobalAveragePooling2D(name=None if name is None else name+"_gap")(x)
    s = tf.keras.layers.Dense(max(channels // ratio, 8), activation="gelu",
                              name=None if name is None else name+"_fc1")(s)
    s = tf.keras.layers.Dense(channels, activation="sigmoid",
                              name=None if name is None else name+"_fc2")(s)
    s = tf.keras.layers.Reshape((1, 1, channels))(s)
    return tf.keras.layers.Multiply(name=None if name is None else name+"_scale")([x, s])

def residual_block(x, filters, stride=1, name="res"):
    shortcut = x

    x = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding="same",
                               use_bias=False, name=name+"_conv1")(x)
    x = tf.keras.layers.BatchNormalization(name=name+"_bn1")(x)
    x = tf.keras.layers.Activation("gelu", name=name+"_act1")(x)

    x = tf.keras.layers.Conv2D(filters, 3, padding="same",
                               use_bias=False, name=name+"_conv2")(x)
    x = tf.keras.layers.BatchNormalization(name=name+"_bn2")(x)
    x = se_block(x, name=name+"_se")

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=stride,
                                           padding="same", use_bias=False,
                                           name=name+"_proj")(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=name+"_proj_bn")(shortcut)

    x = tf.keras.layers.Add(name=name+"_add")([x, shortcut])
    x = tf.keras.layers.Activation("gelu", name=name+"_out")(x)
    return x

def make_branch(input_tensor, prefix):
    x = tf.keras.layers.Conv2D(32, 5, strides=2, padding="same",
                               use_bias=False, name=prefix+"_stem")(input_tensor)
    x = tf.keras.layers.BatchNormalization(name=prefix+"_stem_bn")(x)
    x = tf.keras.layers.Activation("gelu")(x)

    x = residual_block(x, 32, 1, prefix+"_r1")
    x = residual_block(x, 64, 2, prefix+"_r2")
    x = residual_block(x, 64, 1, prefix+"_r3")
    x = residual_block(x, 128, 2, prefix+"_r4")
    x = residual_block(x, 128, 1, prefix+"_r5")
    x = residual_block(x, 192, 2, prefix+"_r6")

    x = tf.keras.layers.GlobalAveragePooling2D(name=prefix+"_gap")(x)
    x = tf.keras.layers.Dense(192, activation="gelu", name=prefix+"_embedding")(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    return x

eeg_input = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="eeg_image")
gsr_input = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="gsr_image")

# Very light augmentation only; no flips because time/frequency orientation matters.
eeg_aug = tf.keras.Sequential([
    tf.keras.layers.RandomTranslation(0.01, 0.015),
    tf.keras.layers.RandomZoom(0.02, 0.02),
], name="eeg_aug")

gsr_aug = tf.keras.Sequential([
    tf.keras.layers.RandomTranslation(0.01, 0.015),
    tf.keras.layers.RandomZoom(0.02, 0.02),
], name="gsr_aug")

eeg_feat = make_branch(eeg_aug(eeg_input), "eeg")
gsr_feat = make_branch(gsr_aug(gsr_input), "gsr")

# Gated modality fusion
concat = tf.keras.layers.Concatenate(name="concat_features")([eeg_feat, gsr_feat])

gate = tf.keras.layers.Dense(2, activation="softmax", name="modality_gate")(concat)
eeg_gate = tf.keras.layers.Lambda(lambda z: z[:, 0:1], name="eeg_gate")(gate)
gsr_gate = tf.keras.layers.Lambda(lambda z: z[:, 1:2], name="gsr_gate")(gate)

eeg_weighted = tf.keras.layers.Multiply()([eeg_feat, eeg_gate])
gsr_weighted = tf.keras.layers.Multiply()([gsr_feat, gsr_gate])

fusion = tf.keras.layers.Concatenate(name="gated_fusion")([eeg_weighted, gsr_weighted])

x = tf.keras.layers.Dense(256, activation="gelu")(fusion)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.30)(x)
x = tf.keras.layers.Dense(128, activation="gelu")(x)
x = tf.keras.layers.Dropout(0.20)(x)

output = tf.keras.layers.Dense(3, activation="softmax", name="stress_output")(x)

model = tf.keras.Model(
    [eeg_input, gsr_input],
    output,
    name="StressV3_DualCNN_GatedFusion"
)

model.summary()

# ------------------------------------------------------------
# Compile / train
# ------------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

BEST = MODEL_DIR / "best.weights.h5"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(BEST),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=3,
        factor=0.4,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# ------------------------------------------------------------
# Curves
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("V3 Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("V3 Loss")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("STRICT UNSEEN-SUBJECT TEST")
print("=" * 76)

test_loss, test_acc_keras = model.evaluate(test_ds, verbose=1)
probs = model.predict(test_ds, verbose=1)
pred = np.argmax(probs, axis=1)
y = test_df["label_id"].values.astype(np.int32)

acc = accuracy_score(y, pred)
macro_p = precision_score(y, pred, average="macro", zero_division=0)
macro_r = recall_score(y, pred, average="macro", zero_division=0)
macro_f1 = f1_score(y, pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y, pred, average="weighted", zero_division=0)
recalls = recall_score(y, pred, labels=[0,1,2], average=None, zero_division=0)

try:
    onehot = tf.keras.utils.to_categorical(y, 3)
    auc = roc_auc_score(onehot, probs, multi_class="ovr", average="macro")
except Exception:
    auc = np.nan

cm = confusion_matrix(y, pred, labels=[0,1,2])

print("\nFINAL V3 RESULTS")
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {macro_p:.4f}")
print(f"Macro Recall    : {macro_r:.4f}")
print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")
print(f"Macro ROC-AUC   : {auc:.4f}")
print(f"\nLow Recall      : {recalls[0]:.4f}")
print(f"Moderate Recall : {recalls[1]:.4f}")
print(f"High Recall     : {recalls[2]:.4f}")

print("\nClassification Report:\n")
print(classification_report(y, pred, target_names=LABELS, zero_division=0))

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(7,6))
plt.imshow(cm)
plt.xticks(range(3), LABELS)
plt.yticks(range(3), LABELS)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("V3 Confusion Matrix")
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

# Save predictions
pred_df = test_df.copy()
pred_df["predicted_label_id"] = pred
pred_df["predicted_label"] = [LABELS[i] for i in pred]
pred_df["prob_low"] = probs[:,0]
pred_df["prob_moderate"] = probs[:,1]
pred_df["prob_high"] = probs[:,2]
pred_df.to_csv(MODEL_DIR / "test_predictions.csv", index=False)

# Save model
KERAS_PATH = MODEL_DIR / "stress_v3_dualcnn.keras"
model.save(KERAS_PATH)

print("\nSaved Keras model:", KERAS_PATH)

# Keep variables for optional ensemble cell.
V3_SINGLE_MODEL_ACCURACY = float(acc)
V3_SINGLE_MODEL_F1 = float(macro_f1)


In [ ]:

# ============================================================
# 5. OPTIONAL: 3-SEED ENSEMBLE
#    Run this only if the single V3 model is below your target.
# ============================================================

# This cell trains 2 additional copies with different random seeds
# and averages the probabilities of all 3 models.
#
# It can improve stability/generalization, but it does NOT guarantee 80%.

if V3_SINGLE_MODEL_ACCURACY >= 0.80:
    print("Single model already reached >= 80%. Ensemble is optional.")
else:
    print("Single model below 80%; training 2 additional ensemble members...")

    ensemble_probs = [probs]
    ensemble_paths = [str(KERAS_PATH)]

    for extra_seed in [123, 777]:
        tf.keras.backend.clear_session()
        random.seed(extra_seed)
        np.random.seed(extra_seed)
        tf.random.set_seed(extra_seed)

        # Rebuild exactly the same architecture.
        eeg_input2 = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="eeg_image")
        gsr_input2 = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="gsr_image")

        eeg_feat2 = make_branch(eeg_input2, f"eeg_{extra_seed}")
        gsr_feat2 = make_branch(gsr_input2, f"gsr_{extra_seed}")

        concat2 = tf.keras.layers.Concatenate()([eeg_feat2, gsr_feat2])
        gate2 = tf.keras.layers.Dense(2, activation="softmax")(concat2)

        eg2 = tf.keras.layers.Lambda(lambda z: z[:,0:1])(gate2)
        gg2 = tf.keras.layers.Lambda(lambda z: z[:,1:2])(gate2)

        ew2 = tf.keras.layers.Multiply()([eeg_feat2, eg2])
        gw2 = tf.keras.layers.Multiply()([gsr_feat2, gg2])

        z = tf.keras.layers.Concatenate()([ew2, gw2])
        z = tf.keras.layers.Dense(256, activation="gelu")(z)
        z = tf.keras.layers.BatchNormalization()(z)
        z = tf.keras.layers.Dropout(0.30)(z)
        z = tf.keras.layers.Dense(128, activation="gelu")(z)
        z = tf.keras.layers.Dropout(0.20)(z)
        out2 = tf.keras.layers.Dense(3, activation="softmax")(z)

        m = tf.keras.Model([eeg_input2, gsr_input2], out2)

        m.compile(
            optimizer=tf.keras.optimizers.AdamW(
                learning_rate=3e-4,
                weight_decay=1e-4
            ),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        cb = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=7,
                restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", patience=3,
                factor=0.4, min_lr=1e-6, verbose=1
            )
        ]

        m.fit(
            train_ds,
            validation_data=val_ds,
            epochs=35,
            class_weight=class_weights,
            callbacks=cb,
            verbose=1
        )

        p = m.predict(test_ds, verbose=1)
        ensemble_probs.append(p)

        kp = MODEL_DIR / f"stress_v3_seed_{extra_seed}.keras"
        m.save(kp)
        ensemble_paths.append(str(kp))

    avg_probs = np.mean(np.stack(ensemble_probs, axis=0), axis=0)
    ens_pred = np.argmax(avg_probs, axis=1)

    ens_acc = accuracy_score(y, ens_pred)
    ens_f1 = f1_score(y, ens_pred, average="macro", zero_division=0)
    ens_recalls = recall_score(y, ens_pred, labels=[0,1,2], average=None, zero_division=0)
    ens_cm = confusion_matrix(y, ens_pred)

    print("\n" + "="*76)
    print("3-MODEL ENSEMBLE RESULTS")
    print("="*76)
    print(f"Accuracy        : {ens_acc:.4f}")
    print(f"Macro F1        : {ens_f1:.4f}")
    print(f"Low Recall      : {ens_recalls[0]:.4f}")
    print(f"Moderate Recall : {ens_recalls[1]:.4f}")
    print(f"High Recall     : {ens_recalls[2]:.4f}")
    print("\nConfusion Matrix:")
    print(ens_cm)

    if ens_acc >= 0.80:
        print("\n✅ Ensemble crossed the 80% target on the strict unseen-subject test.")
    else:
        print("\n⚠️ Still below 80%. Do not add subject leakage; the dataset/labels are then the limiting factor.")


In [ ]:

# ============================================================
# 6. EXPORT SINGLE V3 MODEL TO TFLITE
# ============================================================

import os
import tensorflow as tf
from pathlib import Path

MODEL_DIR = Path("/content/stress_model_v3")
KERAS_PATH = MODEL_DIR / "stress_v3_dualcnn.keras"
TFLITE_PATH = MODEL_DIR / "stress_v3_dualcnn.tflite"

model_for_export = tf.keras.models.load_model(KERAS_PATH, safe_mode=False)

converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

try:
    tflite = converter.convert()
except Exception as e:
    print("Optimized conversion failed:", e)
    print("Retrying standard conversion...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
    tflite = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite)

size_mb = os.path.getsize(TFLITE_PATH) / 1024**2

print("TFLite:", TFLITE_PATH)
print("Size MB:", round(size_mb, 2))
print("OFFLINE EXPORT COMPLETE ✅")


In [ ]:

# ============================================================
# 7. PACKAGE RESULTS FOR DOWNLOAD
# ============================================================

import shutil
from pathlib import Path

src = Path("/content/stress_model_v3")
zip_path = shutil.make_archive(
    "/content/stress_model_v3_results",
    "zip",
    root_dir=src
)

print("Results ZIP:", zip_path)
print("You can download this ZIP from the Colab Files panel.")
